In [ ]:
import torch
from torch.utils.data import DataLoader
from features.build_features import TrialDataset
from models.trial_success_model import TrialSuccessModel
from evaluation.metrics import compute_metrics
from evaluation.plots import plot_confusion_matrix, plot_roc_curve

# Load dataset
data_csv = "../data/trials.csv"
dataset = TrialDataset(data_csv)
loader = DataLoader(dataset, batch_size=16, shuffle=False)

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TrialSuccessModel().to(device)
model.load_state_dict(torch.load("../models/checkpoint.pt", map_location=device))
model.eval()

# Evaluate
all_preds, all_targets = [], []
with torch.no_grad():
    for batch in loader:
        inputs, targets = batch
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        all_preds.extend(outputs.squeeze().cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

# Compute metrics
metrics = compute_metrics(all_targets, all_preds)
print("Evaluation Metrics:", metrics)

# Plots
plot_confusion_matrix(all_targets, all_preds, threshold=0.5)
plot_roc_curve(all_targets, all_preds)